In [1]:
import pandas as pd
import numpy as np
import joblib
import glob
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

# 1. LOAD AND MERGE PARQUET FILES
print("[*] Reading parquet files from data/archive...")
file_paths = glob.glob('../data/archive/*.parquet')

df_list = []
for file in file_paths:
    temp_df = pd.read_parquet(file)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
print(f"[*] Total rows: {len(df):,}")
print(f"[*] Columns: {df.columns.tolist()}")

# 2. LABEL PROCESSING (CTU-13 specific)
def categorize_label(label_str):
    label_str = str(label_str).lower()
    if 'botnet' in label_str:
        return 1  # Malicious
    elif 'normal' in label_str:
        return 0  # Benign
    else:
        return -1  # Background (to be dropped)

df['Target'] = df['label'].apply(categorize_label)

# Keep only Botnet (1) and Normal (0), drop Background (-1)
df = df[df['Target'] != -1]
print(f"[*] After dropping background: {len(df):,} rows")

# 3. FEATURE SELECTION
features = [
    'dur',
    'tot_pkts',
    'tot_bytes',
    'src_bytes',
    'proto',
    'state'
]

X = df[features].copy()
y = df['Target'].copy()

# Drop NaN
valid_indices = X.dropna().index
X = X.loc[valid_indices]
y = y.loc[valid_indices]

# Encode categorical features
le_proto = LabelEncoder()
X['proto'] = le_proto.fit_transform(X['proto'].astype(str))

le_state = LabelEncoder()
X['state'] = le_state.fit_transform(X['state'].astype(str))

# 4. BALANCE DATA (undersampling)
print(f"[*] Distribution before balancing: \n{y.value_counts()}")
botnet_indices = y[y == 1].index
normal_indices = y[y == 0].index

min_samples = min(len(botnet_indices), len(normal_indices))
random_normal_indices = np.random.choice(normal_indices, min_samples, replace=False)
random_botnet_indices = np.random.choice(botnet_indices, min_samples, replace=False)

balanced_indices = np.concatenate([random_botnet_indices, random_normal_indices])
X_balanced = X.loc[balanced_indices]
y_balanced = y.loc[balanced_indices]

print(f"[*] Distribution after balancing: \n{y_balanced.value_counts()}")

# 5. TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced
)

# 6. TRAIN XGBOOST
print("[*] Training XGBoost classifier...")
xgb_clf = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1,
    eval_metric='logloss'
)
xgb_clf.fit(X_train, y_train)

# 7. EVALUATE
y_pred = xgb_clf.predict(X_test)
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

print("\n[+] EVALUATION RESULTS:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Botnet (1)']))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

# 8. SAVE MODEL
joblib.dump(xgb_clf, '../data/surrogate_ids_ctu13.pkl')
joblib.dump(le_proto, '../data/label_encoder_proto.pkl')
joblib.dump(le_state, '../data/label_encoder_state.pkl')
print("[+] Model and encoders saved successfully!")

[*] Reading parquet files from data/archive...


[*] Total rows: 10,598,771
[*] Columns: ['dur', 'proto', 'dir', 'state', 'stos', 'dtos', 'tot_pkts', 'tot_bytes', 'src_bytes', 'label', 'Family']


[*] After dropping background: 465,122 rows


[*] Distribution before balancing: 
Target
1    262504
0    202549
Name: count, dtype: int64


[*] Distribution after balancing: 
Target
1    202549
0    202549
Name: count, dtype: int64
[*] Training XGBoost classifier...



[+] EVALUATION RESULTS:
              precision    recall  f1-score   support

  Normal (0)       0.91      0.92      0.91     40510
  Botnet (1)       0.92      0.91      0.91     40510

    accuracy                           0.91     81020
   macro avg       0.91      0.91      0.91     81020
weighted avg       0.91      0.91      0.91     81020

ROC AUC: 0.9717
[+] Model and encoders saved successfully!


In [2]:
# Extract malicious dataset for the RL environment
malicious_df = df[df['Target'] == 1].copy()

print(f"[+] Malicious samples: {len(malicious_df):,}")
print(f"[+] Normal samples:    {len(df[df['Target'] == 0]):,}")

print("\n[*] Malicious label distribution:")
print(malicious_df['label'].value_counts())

# Save malicious dataset
output_path = '../data/malicious_ctu13.parquet'
malicious_df.to_parquet(output_path, index=False)
print(f"\n[+] Malicious dataset saved to: {output_path}")

[+] Malicious samples: 262,573
[+] Normal samples:    202,549

[*] Malicious label distribution:
label
flow=From-Botnet-V42-UDP-DNS                                    25361
flow=From-Botnet-V50-1-UDP-DNS                                  14404
flow=From-Botnet-V44-TCP-Attempt                                12377
flow=From-Botnet-V50-7-UDP-DNS                                  11887
flow=From-Botnet-V50-3-UDP-DNS                                  11180
                                                                ...  
flow=From-Botnet-V48-TCP-HTTP-Google-Net-Established-6              1
flow=From-Botnet-V48-UDP-Attempt                                    1
flow=From-Botnet-V48-TCP-HTTP-Google-Net-Established-7              1
flow=From-Botnet-V48-TCP-Established-HTTP-Binary-Download-12        1
flow=From-Botnet-V48-TCP-Established-HTTP-Ad-4                      1
Name: count, Length: 1263, dtype: int64

[+] Malicious dataset saved to: ../data/malicious_ctu13.parquet
